In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import shap
import xgboost as xgb


# PART 1 — SYNTHETIC COMPANY DATA


np.random.seed(42)

n = 1000

data = pd.DataFrame({
    "revenue": np.random.normal(1_000_000, 200_000, n),
    "cogs": np.random.normal(400_000, 100_000, n),
    "officer_comp": np.random.normal(150_000, 50_000, n),
    "meals": np.random.normal(25_000, 10_000, n),
    "travel": np.random.normal(40_000, 15_000, n),
})

data["total_deductions"] = data["cogs"] + data["officer_comp"] + data["meals"] + data["travel"]
data["net_income"] = data["revenue"] - data["total_deductions"]

# Feature engineering
data["meals_ratio"] = data["meals"] / data["revenue"]
data["officer_ratio"] = data["officer_comp"] / data["revenue"]

# Create synthetic audit flag
data["audit_flag"] = (
    (data["meals_ratio"] > 0.06) |
    (data["officer_ratio"] > 0.25)
).astype(int)

# PART 2 — AUDIT RISK MODEL


features = ["revenue", "cogs", "officer_comp", "meals", "travel",
            "meals_ratio", "officer_ratio"]

X = data[features]
y = data["audit_flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

risk_model = xgb.XGBClassifier()
risk_model.fit(X_train, y_train)

print("\nAudit Risk Model Accuracy:")
print(risk_model.score(X_test, y_test))


# PART 3 — ANOMALY DETECTION


iso = IsolationForest(contamination=0.05)
data["anomaly_score"] = iso.fit_predict(X)

# -1 means anomaly
print("\nNumber of flagged anomalies:")
print((data["anomaly_score"] == -1).sum())


# PART 4 — NLP TRANSACTION CLASSIFIER


descriptions = [
    "Client dinner at restaurant",
    "Flight ticket for conference",
    "CEO annual salary payment",
    "Hotel stay for business trip",
    "Executive bonus payout"
]

labels = ["meals", "travel", "officer_comp", "travel", "officer_comp"]

vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(descriptions)

clf = LogisticRegression()
clf.fit(X_text, labels)

test_text = ["Business lunch with investor"]
test_vector = vectorizer.transform(test_text)

print("\nPredicted category for test transaction:")
print(clf.predict(test_vector))


# PART 5 — EXPLAINABILITY (SHAP)


explainer = shap.Explainer(risk_model)
shap_values = explainer(X_test[:50])

print("\nGenerating SHAP explanation summary plot...")
shap.summary_plot(shap_values, X_test[:50])